# Rara 7B Fine-tune (Colab T4 GPU, ~10 menit total)

Notebook ini gantiin workflow GitHub Actions yang lambat (CPU 7B = 39 menit) dan sering kena shutdown signal. Colab free kasih **T4 GPU 16 GB** yang 30-50x lebih cepat buat 7B QLoRA.

**Cara pakai:**
1. Buka https://colab.research.google.com
2. File → Upload notebook → pilih file ini
3. Runtime → Change runtime type → T4 GPU
4. Di cell #8, ganti `HF_TOKEN = "..."` dengan token Write dari https://huggingface.co/settings/tokens
5. Runtime → Run all

**Expected:** train ~1 menit, save ~2 menit, GGUF convert ~2 menit, HF upload ~3 menit. Total ~10 menit.

In [ ]:
# 1. Cek GPU (harus T4)
import torch
assert torch.cuda.is_available(), 'GPU gak ada! Runtime → Change runtime type → T4 GPU'
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
import subprocess
print('CPU:', subprocess.check_output(['nproc']).decode().strip(), 'cores')

In [ ]:
# 2. Install dependencies
!pip install -q transformers datasets peft accelerate bitsandbytes sentencepiece huggingface_hub
import torch, transformers, peft, bitsandbytes
print(f'torch {torch.__version__}, transformers {transformers.__version__}, peft {peft.__version__}')

In [ ]:
# 3. Download dataset dari repo rara-agent
import urllib.request, json
url = 'https://raw.githubusercontent.com/Yz776/rara-agent/main/training-data/rara_training_v2.jsonl'
urllib.request.urlretrieve(url, '/content/rara_training_v2.jsonl')
with open('/content/rara_training_v2.jsonl') as f:
    lines = f.readlines()
print(f'Dataset: {len(lines)} examples')
for i, line in enumerate(lines[:3]):
    d = json.loads(line)
    assert 'messages' in d, f'Line {i}: no messages'
print('Format OK!')

In [ ]:
# 4. Load Qwen2.5-7B-Instruct in 4-bit QLoRA (same as GH Actions workflow)
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training

MODEL = 'Qwen/Qwen2.5-7B-Instruct'
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,  # GPU-safe bf16
    bnb_4bit_use_double_quant=True,
)
tokenizer = AutoTokenizer.from_pretrained(MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
    low_cpu_mem_usage=True,
)
model = prepare_model_for_kbit_training(model)

lora = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
)
model = get_peft_model(model, lora)
model.enable_input_require_grads()
model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant': False})

n_t = sum(p.numel() for p in model.parameters() if p.requires_grad)
n_a = sum(p.numel() for p in model.parameters())
print(f'Trainable: {n_t:,} / {n_a:,} ({100*n_t/n_a:.1f}%)')

In [ ]:
# 5. Format dataset (apply chat template)
from datasets import load_dataset
ds = load_dataset('json', data_files='/content/rara_training_v2.jsonl', split='train')
SYS = 'Kamu Rara. Pakai Aku bukan Saya, kamu bukan Anda. Jawab singkat + ceria + emoji. Bahasa Indonesia santai.'

def fmt(x):
    msgs = [dict(m) for m in x['messages']]
    if msgs and msgs[0]['role'] == 'system':
        msgs[0]['content'] = SYS
    text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)
    tokens = tokenizer(text, truncation=True, max_length=1024, return_tensors=None)
    tokens['labels'] = tokens['input_ids'].copy()
    return tokens

ds = ds.map(fmt, remove_columns=ds.column_names)
print(f'Dataset: {len(ds)} examples')

In [ ]:
# 6. Train (~1-3 menit di T4 GPU)
from transformers import TrainingArguments, Trainer, DataCollatorForSeq2Seq
import inspect

args = TrainingArguments(
    output_dir='/content/rara-ckpt',
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    max_steps=20,
    learning_rate=2e-4,
    warmup_steps=2,
    logging_steps=1,
    save_steps=10,
    save_total_limit=1,
    report_to='none',
    weight_decay=0.01,
    max_grad_norm=1.0,
    seed=42,
    lr_scheduler_type='cosine',
    fp16=False,
    bf16=True,  # T4 supports bf16
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False},
    dataloader_num_workers=2,
    remove_unused_columns=False,
    logging_first_step=True,
)

collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, padding=True)
_kwargs = dict(model=model, args=args, train_dataset=ds, data_collator=collator)
if 'processing_class' in inspect.signature(Trainer).parameters:
    _kwargs['processing_class'] = tokenizer
else:
    _kwargs['tokenizer'] = tokenizer
trainer = Trainer(**_kwargs)

print('Training started (T4 GPU mode)...')
stats = trainer.train()
print(f'Done! Steps={stats.global_step} Loss={stats.training_loss:.4f}')

In [ ]:
# 7. Merge LoRA + dequantize 4-bit weights to float16
# (Wajib: convert_hf_to_gguf.py gak bisa baca model yang masih 4-bit)
import torch.nn as nn
import bitsandbytes as bnb

def _dequant_recursive(parent_module):
    for name, child in list(parent_module.named_children()):
        if isinstance(child, bnb.nn.Linear4bit) and hasattr(child.weight, 'quant_state'):
            dequant_w = bnb.functional.dequantize_4bit(
                child.weight.data, child.weight.quant_state
            ).to(torch.float16)
            new_linear = nn.Linear(
                child.in_features, child.out_features,
                bias=child.bias is not None,
            )
            new_linear.weight = nn.Parameter(dequant_w)
            if child.bias is not None:
                new_linear.bias = nn.Parameter(child.bias.data.to(torch.float16))
            setattr(parent_module, name, new_linear)
        else:
            _dequant_recursive(child)

model = model.merge_and_unload()
_dequant_recursive(model)
model = model.to(torch.float16).to('cpu')
if hasattr(model, 'config') and hasattr(model.config, 'quantization_config'):
    model.config.quantization_config = None
model.save_pretrained('/content/rara-model', safe_serialization=True)
tokenizer.save_pretrained('/content/rara-model')

import os
size = sum(os.path.getsize(os.path.join(d,f)) for d,_,fs in os.walk('/content/rara-model') for f in fs)
print(f'Model saved! Size: {size/1e9:.2f} GB')

In [ ]:
# 8. Set HuggingFace token (GANTI dengan token Write milik lo!)
# Bisa juga pake Colab secret: from google.colab import userdata; HF_TOKEN = userdata.get('HF_TOKEN')
HF_TOKEN = 'PASTE_HF_TOKEN_HERE'  # <-- GANTI INI

from huggingface_hub import HfApi
api = HfApi(token=HF_TOKEN)
try:
    info = api.whoami()
    username = info['name']
    print(f'Logged in as: {username}')
except Exception as e:
    print(f'ERROR: {e}')
    print('Buat token Write di https://huggingface.co/settings/tokens')
    raise

In [ ]:
# 9. Clone llama.cpp + convert GGUF (q8_0, in-script quantization)
import subprocess, os
if not os.path.exists('/content/llama.cpp'):
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/ggerganov/llama.cpp', '/content/llama.cpp'], check=True)
subprocess.run(['python', 'convert_hf_to_gguf.py', '/content/rara-model',
                '--outtype', 'q8_0', '--outfile', '/content/rara-q8_0.gguf'],
                cwd='/content/llama.cpp', check=True)
size = os.path.getsize('/content/rara-q8_0.gguf') / 1e9
print(f'GGUF created! Size: {size:.2f} GB')

In [ ]:
# 10. Upload ke HuggingFace (public repo: {username}/rara-ai)
from huggingface_hub import create_repo, upload_file
REPO = f'{username}/rara-ai'
print(f'Uploading to: {REPO}')
create_repo(REPO, token=HF_TOKEN, exist_ok=True, private=False)
upload_file(
    path_or_fileobj='/content/rara-q8_0.gguf',
    path_in_repo='rara-q8_0.gguf',
    repo_id=REPO,
    token=HF_TOKEN,
)
print(f'SUCCESS! -> https://huggingface.co/{REPO}')

## Deploy ke server rara.ai.id

Setelah upload selesai, SSH ke server 209.50.50.121 dan jalankan di kali container:

```bash
docker exec kali ollama pull hf.co/{username}/rara-ai:rara-q8_0.gguf
echo 'MODEL=hf.co/{username}/rara-ai:rara-q8_0.gguf' >> /opt/rara-agent/.env
docker exec kali pm2 restart rara-web
```

(Ganti `{username}` dengan HF username yang keluar di cell #8.)